# Fine-tune Qwen2.5-3B-Instruct for Medical Validator

QLoRA fine-tuning on Google Colab free tier (T4 16GB).

**Steps:**
1. Install dependencies
2. Upload `dataset.jsonl` (from `npx tsx training/export-training-data.ts`)
3. Load base model in 4-bit
4. Train with LoRA adapters
5. Merge & export to GGUF for Ollama

**Runtime:** ~30-60 min for 500 examples, 3 epochs on T4.

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q transformers==4.51.0 datasets peft==0.13.2 trl==0.12.2 bitsandbytes==0.45.0 accelerate==1.2.0
!pip install -q llama-cpp-python  # for GGUF export

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Upload Training Data

Upload `dataset.jsonl` generated by `npx tsx training/export-training-data.ts`

In [ ]:
from google.colab import files
import os

# Upload dataset.jsonl
uploaded = files.upload()
DATASET_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {DATASET_PATH} ({os.path.getsize(DATASET_PATH) / 1024:.1f} KB)")

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files=DATASET_PATH, split='train')
print(f"Total examples: {len(dataset)}")

# Train/eval split (90/10)
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset = split['test']
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

# Preview first example
import json
for msg in train_dataset[0]['messages']:
    print(f"\n--- {msg['role'].upper()} ---")
    print(msg['content'][:200] + '...' if len(msg['content']) > 200 else msg['content'])

## 2. Load Base Model (4-bit Quantized)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# 4-bit quantization config for QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

print(f"Model loaded: {MODEL_ID}")
print(f"Parameters: {model.num_parameters() / 1e9:.1f}B")

## 3. Configure LoRA

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"LoRA config: r={lora_config.r}, alpha={lora_config.lora_alpha}")
print(f"Target modules: {lora_config.target_modules}")

## 4. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./qwen-medical-validator"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
)

# Format dataset into chat template
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_formatted = train_dataset.map(format_chat)
eval_formatted = eval_dataset.map(format_chat)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_formatted,
    eval_dataset=eval_formatted,
    peft_config=lora_config,
    max_seq_length=2048,
)

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M")
print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Training metrics
import json
metrics = trainer.state.log_history
train_losses = [m['loss'] for m in metrics if 'loss' in m]
eval_losses = [m['eval_loss'] for m in metrics if 'eval_loss' in m]

print(f"Final train loss: {train_losses[-1]:.4f}")
if eval_losses:
    print(f"Final eval loss:  {eval_losses[-1]:.4f}")

## 5. Quick Validation

Test the fine-tuned model on a sample input before exporting.

In [ ]:
# Quick test with a sample from eval set
model.config.use_cache = True

test_example = eval_dataset[0]['messages']
test_messages = [msg for msg in test_example if msg['role'] != 'assistant']

input_text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=True,
    )

generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print("=== Generated ===")
try:
    parsed = json.loads(generated)
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError:
    print(generated)

print("\n=== Expected ===")
expected = json.loads(test_example[-1]['content'])
print(json.dumps(expected, indent=2))

## 6. Merge & Export

Merge LoRA adapters into base model and save.

In [ ]:
# Save LoRA adapters
trainer.save_model(f"{OUTPUT_DIR}/lora-adapters")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora-adapters")
print(f"LoRA adapters saved to {OUTPUT_DIR}/lora-adapters")

# Free training model from GPU before loading FP16
del trainer
del model
torch.cuda.empty_cache()
import gc
gc.collect()

# Merge into full model
from transformers import AutoModelForCausalLM
from peft import PeftModel

# Reload base model in FP16 for merging
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

merged_model = PeftModel.from_pretrained(base_model, f"{OUTPUT_DIR}/lora-adapters")
merged_model = merged_model.merge_and_unload()

MERGED_DIR = f"{OUTPUT_DIR}/merged"
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}")

## 7. Convert to GGUF (for Ollama)

Convert the merged model to GGUF format so it can be loaded by Ollama.

In [ ]:
# Install llama.cpp conversion tools
!pip install -q huggingface_hub[hf_xet]
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /tmp/llama.cpp
!pip install -q -r /tmp/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

In [ ]:
GGUF_PATH = f"{OUTPUT_DIR}/medical-validator-q4_k_m.gguf"

!python /tmp/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
    --outfile {GGUF_PATH} \
    --outtype q4_k_m

import os
size_mb = os.path.getsize(GGUF_PATH) / (1024 * 1024)
print(f"\nGGUF exported: {GGUF_PATH} ({size_mb:.0f} MB)")

In [ ]:
# Create Ollama Modelfile
with open(f"{OUTPUT_DIR}/Modelfile", "w") as f:
    f.write('FROM ./medical-validator-q4_k_m.gguf\n\n')
    f.write('PARAMETER temperature 0\n')
    f.write('PARAMETER num_ctx 2048\n')
    f.write('PARAMETER stop <|im_end|>\n\n')
    f.write('TEMPLATE """{{ if .System }}<|im_start|>system\n')
    f.write('{{ .System }}<|im_end|>\n')
    f.write('{{ end }}<|im_start|>user\n')
    f.write('{{ .Prompt }}<|im_end|>\n')
    f.write('<|im_start|>assistant\n')
    f.write('"""\n')

print("Modelfile created.")
print("\nTo load in Ollama:")
print(f"  cd {OUTPUT_DIR}")
print("  ollama create medical-validator -f Modelfile")
print("  ollama run medical-validator")

In [ ]:
# Download the GGUF + Modelfile
from google.colab import files

# Zip for easier download
!cd {OUTPUT_DIR} && zip -j medical-validator-ollama.zip medical-validator-q4_k_m.gguf Modelfile
files.download(f"{OUTPUT_DIR}/medical-validator-ollama.zip")
print("\nDownload started. Extract and run 'ollama create medical-validator -f Modelfile'")